# Ge 158 Problem Set 3, Problem 2: starter notebook (Eaton Fire)
Python version of `ge158_2026_pset3_p2_starter.js`. It does the same calculations; use whichever you prefer. Use the layer control (top right of the map) to turn layers on and off.

**Setup.** This notebook runs in Google Colab (File > Open notebook > Upload), or in Jupyter with `earthengine-api` and `geemap` installed. Set `PROJECT` to the Google Cloud project you used for Problem Set 2. The first time you run the setup cell, it asks you to sign in to your Google account.

In [ ]:
# If geemap is missing (e.g., in a local Jupyter), uncomment the next line
# %pip install -q earthengine-api geemap
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = 'your-cloud-project-id'  # the Cloud project from Problem Set 2
ee.Authenticate()
ee.Initialize(project=PROJECT)

In [ ]:
roi = ee.Geometry.Rectangle([-118.20, 34.15, -117.98, 34.27])
Map = geemap.Map()
Map.centerObject(roi, 12)

## Part (a): MODIS burned area for January 2025
`BurnDate` is the day of the year of the burn; 0 means unburned and is masked here.

In [ ]:
burn_date = (ee.ImageCollection('MODIS/061/MCD64A1')
             .filterDate('2025-01-01', '2025-02-01').first()
             .select('BurnDate').selfMask())
Map.addLayer(burn_date, {'min': 1, 'max': 31,
                         'palette': ['yellow', 'orange', 'red', 'purple']},
             'MODIS burn date (adjust min and max)')

## Part (b): Sentinel-2 surface reflectance before and after the fire
Mosaic of all tiles from one overpass; DN / 10000 gives reflectance.

In [ ]:
def s2_mosaic(date):
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(roi)
            .filterDate(date, ee.Date(date).advance(1, 'day'))
            .mosaic().divide(10000))

s2_pre = s2_mosaic('2024-12-18')
s2_post = s2_mosaic('2025-02-21')
Map.addLayer(s2_pre, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3},
             'Sentinel-2 before (18 Dec 2024)')
Map.addLayer(s2_post, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3},
             'Sentinel-2 after (21 Feb 2025)')
# Hint: image.normalizedDifference(['B8', 'B4']) computes
# (B8 - B4) / (B8 + B4).

## Part (c): Sentinel-1 backscatter (already in dB)
Median of all images from one descending orbit (relative orbit 71) before and after the fire, and the change, post minus pre.

In [ ]:
s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(roi)
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.eq('relativeOrbitNumber_start', 71))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
      .select(['VV', 'VH']))
s1_pre = s1.filterDate('2024-11-15', '2025-01-07').median()
s1_post = s1.filterDate('2025-01-15', '2025-03-01').median()
s1_change = s1_post.subtract(s1_pre)  # dB
change_vis = {'min': -4, 'max': 4, 'palette': ['blue', 'white', 'red']}
Map.addLayer(s1_change.select('VV'), change_vis, 'Change in VV (dB)')
Map.addLayer(s1_change.select('VH'), change_vis, 'Change in VH (dB)')
Map